In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
import joblib

# Load the data
data = pd.read_csv('preprocessed.csv')

# Define features and target
features = ['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday', 
            'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed', 
            'distance_to_next_stop', 'current_lat', 'current_lon']
target = 'eta_minutes'

X = data[features]
y = data[target]

# Convert boolean columns to int
X['is_holiday'] = X['is_holiday'].astype(int)
X['is_peak_hour'] = X['is_peak_hour'].astype(int)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Function to evaluate model
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2 Score: {r2:.4f}")
    
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Baseline LightGBM model
print("Training baseline LightGBM model...")
baseline_model = lgb.LGBMRegressor(random_state=42)
baseline_model.fit(X_train, y_train)

print("\nBaseline Model Performance:")
baseline_metrics = evaluate_model(baseline_model, X_test, y_test)

# Optuna optimization
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbosity': -1,
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
    }
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    
    return rmse

print("\nStarting Optuna optimization...")
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)

# Train optimized model
print("\nTraining optimized model with best parameters...")
best_params = study.best_params
optimized_model = lgb.LGBMRegressor(**best_params, random_state=42)
optimized_model.fit(X_train, y_train)

print("\nOptimized Model Performance:")
optimized_metrics = evaluate_model(optimized_model, X_test, y_test)

# Compare baseline and optimized models
print("\nPerformance Comparison:")
print(f"Baseline RMSE: {baseline_metrics['RMSE']:.4f}")
print(f"Optimized RMSE: {optimized_metrics['RMSE']:.4f}")
print(f"Improvement: {(baseline_metrics['RMSE'] - optimized_metrics['RMSE']):.4f} ({((baseline_metrics['RMSE'] - optimized_metrics['RMSE'])/baseline_metrics['RMSE']*100):.2f}%)")

# Save the optimized model
model_filename = 'optimized_lightgbm_eta_predictor.pkl'
joblib.dump(optimized_model, model_filename)
print(f"\nOptimized model saved as {model_filename}")

# Feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': optimized_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(importance)

c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\cheng\AppData\Local\Temp\ipykernel_28476\2480288965.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_holiday'] = X['is_holiday'].astype(int)
C:\Users\cheng\AppData\Local\Temp\ipykernel_28476\2480288965.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexin

Training baseline LightGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000676 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 848
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 11
[LightGBM] [Info] Start training from score 2.238962

Baseline Model Performance:
MAE: 0.3270
MSE: 0.2857
RMSE: 0.5345
R2 Score: 0.9657

Starting Optuna optimization...


c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
[I 2025-03-27 17:20:50,198] Trial 0 finished with value: 0.5446758377085138 and parameters: {'n_estimators': 218, 'learning_rate': 0.2536999076681772, 'num_leaves': 225, 'max_depth': 8, 'min_child_samples': 19, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 8.661761457749352, 'reg_lambda': 6.011150117432088}. Best is trial 0 with value: 0.5446758377085138.
c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
[I 2025-03-27 17


Training optimized model with best parameters...

Optimized Model Performance:
MAE: 0.3156
MSE: 0.2832
RMSE: 0.5322
R2 Score: 0.9660

Performance Comparison:
Baseline RMSE: 0.5345
Optimized RMSE: 0.5322
Improvement: 0.0024 (0.44%)

Optimized model saved as optimized_lightgbm_eta_predictor.pkl

Feature Importance:
                  feature  importance
7           current_speed       12111
9             current_lat        8434
6         passenger_count        6993
10            current_lon        6499
4            is_peak_hour        3148
2             day_of_week        2640
0       current_stop_name        1783
5       weather_condition        1465
1          next_stop_name         726
8   distance_to_next_stop         455
3              is_holiday           0
